# BÀI THI GIỮA KỲ HỌC PHẦN HỌC MÁY (MACHINE LEARNING)
- **Họ và tên sinh viên:** Nguyễn Đỗ Thắng
- **Mã số sinh viên (MSSV):** 102230046
- **Lớp:** 23T_NHAT1
- **Đề bài tổng quát:**
  1. Phân đoạn ảnh màu và ảnh trắng đen bằng **K-Means** và **Fuzzy C-Means (FCM)** (thử nghiệm tối thiểu 4-5 cụm, chọn 5 và 7 cụm).
  2. Phân cụm dữ liệu bảng bằng **K-Means**, **FCM** và phân lớp bằng **K-NN**.
  3. **Yêu cầu kỹ thuật bắt buộc:**
     - Tự cài đặt thuật toán from scratch (Custom bằng NumPy), không dùng thư viện có sẵn để thực hiện lõi.
     - So sánh đối chứng kết quả, thời gian, biểu đồ và bảng biểu với thư viện `scikit-learn`.
     - Phải có biểu đồ giải thích cơ sở khoa học **TẠI SAO CHỌN K, CHỌN C** cho từng thuật toán.
     - Phân tích bản chất sự khác nhau giữa phân cụm ảnh màu và ảnh trắng đen.


---
## ⚙️ KHỐI CẤU HÌNH ĐẦU VÀO ĐỀ THI (CHỈ CẦN THAY ĐƯỜNG DẪN TẠI ĐÂY)
> **HƯỚNG DẪN CHO THẮNG KHI THI:**
> - Nếu thầy phát file ảnh: Bạn chỉ cần gán tên/đường dẫn ảnh vào biến `IMAGE_PATH`.
> - Nếu thầy phát file dataset: Bạn chỉ cần gán tên/đường dẫn file (.csv, .xlsx) vào biến `DATASET_PATH`.
> - Sau đó bấm menu **Kernel -> Restart & Run All** là bài làm hoàn tất từ A đến Z!


In [ ]:
# ==============================================================================
# CẤU HÌNH ĐƯỜNG DẪN FILE ĐỀ THI CỦA THẦY
# ==============================================================================
import os

# 1. Đường dẫn file ảnh thầy phát (Ví dụ: 'de_thi_anh.jpg', 'anh_mau.png', ...)
IMAGE_PATH = 'sample_color_image.jpg'

# 2. Đường dẫn file dataset thầy phát (Ví dụ: 'dataset_giua_ky.csv', 'data.xlsx', ...)
# Nếu để None hoặc file không tồn tại, chương trình sẽ tự động dùng dataset chuẩn Iris/mô phỏng
DATASET_PATH = None  

# 3. Tên cột nhãn mục tiêu trong dataset (nếu có, để None thì mặc định lấy cột cuối cùng)
TARGET_COLUMN = None

print("=> Cấu hình hiện tại:")
print(f"   - File ảnh đầu vào: {IMAGE_PATH}")
print(f"   - File dữ liệu bảng: {DATASET_PATH if DATASET_PATH else 'Sử dụng dataset chuẩn tích hợp sẵn'}")


---
## 1. THIẾT LẬP MÔI TRƯỜNG & IMPORT THƯ VIỆN


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Thư viện Scikit-Learn dùng để đối chứng so sánh (Benchmark)
from sklearn.cluster import KMeans as SklearnKMeans
from sklearn.neighbors import KNeighborsClassifier as SklearnKNN
from sklearn.metrics import silhouette_score, accuracy_score, confusion_matrix
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Cấu hình đồ họa hiển thị trực tiếp trong notebook
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 110

print("✅ Đã nạp thành công các thư viện cần thiết!")


---
## 2. CƠ SỞ LÝ THUYẾT: PHÂN BIỆT ẢNH MÀU VÀ ẢNH TRẮNG ĐEN TRONG PHÂN CỤM

| Tiêu chí | Ảnh màu (Color Image - RGB) | Ảnh trắng đen / Mức xám (Grayscale) |
| :--- | :--- | :--- |
| **Không gian biểu diễn** | Tensor 3 chiều: $H \times W \times 3$ | Ma trận 2 chiều: $H \times W$ |
| **Vector đặc trưng của điểm ảnh** | Mỗi pixel là vector 3 chiều: $\vec{p} = [R, G, B]^T \in [0, 255]^3$ | Mỗi pixel là đại lượng vô hướng 1 chiều: $p = I \in [0, 255]$ (độ sáng) |
| **Ma trận dữ liệu đưa vào thuật toán** | Làm phẳng thành ma trận: $X \in \mathbb{R}^{(H \cdot W) \times 3}$ | Làm phẳng thành ma trận: $X \in \mathbb{R}^{(H \cdot W) \times 1}$ |
| **Độ đo khoảng cách Euclidean** | $d(p_1, p_2) = \sqrt{(R_1-R_2)^2 + (G_1-G_2)^2 + (B_1-B_2)^2}$ | $d(p_1, p_2) = \|I_1 - I_2\|$ |
| **Tọa độ tâm cụm $\mu_k$** | Là một màu đại diện trong không gian RGB: $[ar{R}_k, \bar{G}_k, \bar{B}_k]^T$ | Là một mức xám đại diện: $\bar{I}_k$ |
| **Ý nghĩa phân cụm thị giác** | **Lượng tử hóa màu (Color Quantization):** Gom các pixel có sắc độ tương tự thành các vùng màu chủ đạo. | **Phân ngưỡng độ tương phản (Thresholding/Contrast):** Tách ảnh thành các vùng sáng, trung gian và vùng tối/bóng râm. |


---
## 3. CÀI ĐẶT CÁC THUẬT TOÁN CUSTOM (FROM SCRATCH 100% NUMPY THUẦN)
*Theo đúng yêu cầu bắt buộc của giảng viên: không dùng thư viện đóng gói sẵn cho thuật toán chính.*


In [ ]:
# ==============================================================================
# THUẬT TOÁN 1: CUSTOM K-MEANS CLUSTERING (TỪ ĐẦU)
# ==============================================================================
class CustomKMeans:
    def __init__(self, k=5, max_iters=100, tol=1e-4, metric='euclidean', random_state=42):
        self.k = k
        self.max_iters = max_iters
        self.tol = tol
        self.metric = metric
        self.random_state = random_state
        self.centroids = None
        self.labels_ = None
        self.inertia_ = None  # WCSS
        self.n_iter_ = 0

    def _compute_distance(self, X, centroids):
        if self.metric == 'euclidean':
            return np.linalg.norm(X[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)
        elif self.metric == 'manhattan':
            return np.sum(np.abs(X[:, np.newaxis, :] - centroids[np.newaxis, :, :]), axis=2)
        else:
            raise ValueError("Chỉ hỗ trợ 'euclidean' hoặc 'manhattan'")

    def _init_centroids(self, X):
        np.random.seed(self.random_state)
        n_samples = X.shape[0]
        indices = np.random.choice(n_samples, self.k, replace=False)
        return X[indices].astype(float)

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        n_samples = X.shape[0]
        self.centroids = self._init_centroids(X)

        for it in range(self.max_iters):
            self.n_iter_ = it + 1
            # 1. Gán cụm: c(j) = argmin_k ||x_j - mu_k||
            dists = self._compute_distance(X, self.centroids)
            labels = np.argmin(dists, axis=1)

            # 2. Cập nhật tâm cụm: mu_k = mean(x thuộc C_k)
            new_centroids = np.zeros_like(self.centroids)
            for j in range(self.k):
                members = X[labels == j]
                if len(members) > 0:
                    new_centroids[j] = np.mean(members, axis=0)
                else:
                    new_centroids[j] = X[np.random.choice(n_samples)]

            # Kiểm tra hội tụ
            if np.linalg.norm(new_centroids - self.centroids) < self.tol:
                break
            self.centroids = new_centroids

        # Tính WCSS / Inertia
        dists = self._compute_distance(X, self.centroids)
        self.labels_ = np.argmin(dists, axis=1)
        euc_dists = np.linalg.norm(X - self.centroids[self.labels_], axis=1)
        self.inertia_ = float(np.sum(euc_dists ** 2))
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        dists = self._compute_distance(X, self.centroids)
        return np.argmin(dists, axis=1)

    def fit_predict(self, X):
        self.fit(X)
        return self.labels_


# ==============================================================================
# THUẬT TOÁN 2: CUSTOM FUZZY C-MEANS (FCM) (TỪ ĐẦU)
# ==============================================================================
class CustomFCM:
    def __init__(self, c=5, m=2.0, max_iters=100, tol=1e-4, random_state=42):
        self.c = c
        self.m = m
        self.max_iters = max_iters
        self.tol = tol
        self.random_state = random_state
        self.centers = None
        self.u = None  # Ma trận mức độ thuộc (N, c)
        self.labels_ = None
        self.n_iter_ = 0

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        n_samples = X.shape[0]

        np.random.seed(self.random_state)
        # Khởi tạo ma trận U ngẫu nhiên sao cho tổng hàng = 1
        self.u = np.random.dirichlet(np.ones(self.c), size=n_samples)

        for it in range(self.max_iters):
            self.n_iter_ = it + 1
            u_old = self.u.copy()

            # 1. Cập nhật tâm cụm v_i = sum(u_ik^m * x_k) / sum(u_ik^m)
            um = self.u ** self.m
            self.centers = (um.T @ X) / np.sum(um, axis=0)[:, np.newaxis]

            # 2. Tính ma trận khoảng cách d_ij
            dists = np.linalg.norm(X[:, np.newaxis, :] - self.centers[np.newaxis, :, :], axis=2)
            dists = np.fmax(dists, 1e-10)

            # 3. Cập nhật ma trận thuộc u_ik
            power = 2.0 / (self.m - 1.0)
            inv_dists = 1.0 / dists
            inv_dists_power = inv_dists ** power
            self.u = inv_dists_power / np.sum(inv_dists_power, axis=1, keepdims=True)

            if np.max(np.abs(self.u - u_old)) < self.tol:
                break

        self.labels_ = np.argmax(self.u, axis=1)
        return self

    def compute_fpc(self):
        """Fuzzy Partition Coefficient"""
        return float(np.sum(self.u ** 2) / len(self.u))


# ==============================================================================
# THUẬT TOÁN 3: CUSTOM K-NEAREST NEIGHBORS (K-NN) (TỪ ĐẦU)
# ==============================================================================
class CustomKNN:
    def __init__(self, k=5, metric='euclidean'):
        self.k = k
        self.metric = metric
        self.X_train = None
        self.y_train = None
        self.classes_ = None

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        self.classes_ = np.unique(self.y_train)
        return self

    def _compute_distance(self, x):
        if self.metric == 'euclidean':
            return np.linalg.norm(self.X_train - x, axis=1)
        elif self.metric == 'manhattan':
            return np.sum(np.abs(self.X_train - x), axis=1)
        else:
            raise ValueError("Metric không hợp lệ")

    def predict_one(self, x):
        dists = self._compute_distance(x)
        k_indices = np.argsort(dists)[:self.k]
        k_labels = self.y_train[k_indices]
        k_dists = dists[k_indices]

        labels, counts = np.unique(k_labels, return_counts=True)
        max_count = np.max(counts)
        candidates = labels[counts == max_count]

        if len(candidates) == 1:
            return candidates[0]
        else:
            # Xử lý hòa phiếu (tie-break) bằng tổng khoảng cách nhỏ nhất
            cand_dist = {c: np.sum(k_dists[k_labels == c]) for c in candidates}
            return min(cand_dist, key=cand_dist.get)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self.predict_one(x) for x in X])

    def score(self, X, y):
        preds = self.predict(X)
        return np.mean(preds == np.asarray(y))

print("✅ Đã khởi tạo hoàn chỉnh 3 class thuật toán Custom (NumPy thuần)!")


---
# PHẦN A: BÀI TOÁN PHÂN ĐOẠN ẢNH (IMAGE SEGMENTATION)
*Thực hiện trên ảnh màu và ảnh mức xám, phân loại tối thiểu 4-5 cụm (thử nghiệm $K=5$ và $K=7$), so sánh Custom vs Sklearn.*


In [ ]:
# 1. TẢI VÀ TIỀN XỬ LÝ ẢNH TỰ ĐỘNG
def get_image_data(path):
    if path is not None and os.path.exists(path):
        print(f"--> Đang nạp ảnh từ: {path}")
        img = Image.open(path)
    else:
        print(f"--> Không tìm thấy '{path}', tự động tạo ảnh màu mẫu phong phú để kiểm tra!")
        h, w = 150, 200
        xx, yy = np.meshgrid(np.linspace(0, 1, w), np.linspace(0, 1, h))
        r = np.clip(np.sin(xx * np.pi * 1.5) * 255 + 50, 0, 255)
        g = np.clip(np.cos(yy * np.pi * 1.2) * 255 + 60, 0, 255)
        b = np.clip(((xx - 0.5)**2 + (yy - 0.5)**2) * 600, 0, 255)
        img = Image.fromarray(np.stack([r, g, b], axis=-1).astype(np.uint8))
        img.save('sample_color_image.jpg')
    
    # Resize ảnh về kích thước chuẩn (200-250px) để tính toán tức thì trong phòng thi
    img.thumbnail((220, 220), Image.Resampling.LANCZOS)
    color_arr = np.array(img.convert('RGB'), dtype=float)
    gray_arr = np.array(img.convert('L'), dtype=float)
    return color_arr, gray_arr

img_color, img_gray = get_image_data(IMAGE_PATH)
h, w, _ = img_color.shape

# Hiển thị 2 ảnh gốc
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(img_color.astype('uint8'))
ax[0].set_title(f"Ảnh màu gốc (RGB)\nKích thước: {w}x{h} (3 kênh)", fontweight='bold')
ax[0].axis('off')

ax[1].imshow(img_gray.astype('uint8'), cmap='gray')
ax[1].set_title(f"Ảnh mức xám gốc (Grayscale)\nKích thước: {w}x{h} (1 kênh)", fontweight='bold')
ax[1].axis('off')
plt.suptitle("HÌNH ẢNH ĐẦU VÀO ĐƯỢC CHUẨN BỊ CHO BÀI THI", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 2. PHÂN ĐOẠN ẢNH MÀU VỚI K=5 VÀ K=7 (CUSTOM K-MEANS VS SKLEARN K-MEANS)
pixels_color = img_color.reshape(-1, 3)
color_records = []

for k in [5, 7]:
    # Custom K-Means
    t0 = time.time()
    c_km = CustomKMeans(k=k, max_iters=50, random_state=42)
    c_km.fit(pixels_color)
    t_c = time.time() - t0
    c_seg = c_km.centroids[c_km.labels_].reshape(h, w, 3).astype('uint8')
    
    # Sklearn K-Means
    t0 = time.time()
    s_km = SklearnKMeans(n_clusters=k, max_iter=50, random_state=42, n_init=5)
    s_km.fit(pixels_color)
    t_s = time.time() - t0
    s_seg = s_km.cluster_centers_[s_km.labels_].reshape(h, w, 3).astype('uint8')
    
    color_records.append({
        'Số cụm K': k,
        'Thời gian Custom (s)': f"{t_c:.3f} s",
        'Thời gian Sklearn (s)': f"{t_s:.3f} s",
        'WCSS Custom': f"{c_km.inertia_:.2e}",
        'WCSS Sklearn': f"{s_km.inertia_:.2e}",
        'Độ lệch tâm cụm': f"{np.mean(np.abs(np.sort(c_km.centroids, axis=0) - np.sort(s_km.cluster_centers_, axis=0))):.4f}"
    })
    
    # Vẽ đối sánh 3 ảnh
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    axes[0].imshow(img_color.astype('uint8'))
    axes[0].set_title("Ảnh màu gốc", fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(c_seg)
    axes[1].set_title(f"Custom K-Means (K={k})\nThời gian: {t_c:.3f}s | WCSS: {c_km.inertia_:.2e}", fontweight='bold', color='navy')
    axes[1].axis('off')
    
    axes[2].imshow(s_seg)
    axes[2].set_title(f"Sklearn K-Means (K={k})\nThời gian: {t_s:.3f}s | WCSS: {s_km.inertia_:.2e}", fontweight='bold', color='darkgreen')
    axes[2].axis('off')
    
    plt.suptitle(f"SO SÁNH PHÂN ĐOẠN ẢNH MÀU (RGB) VỚI K = {k}", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("--- BẢNG ĐỐI SÁNH KẾT QUẢ PHÂN ĐOẠN ẢNH MÀU ---")
display(pd.DataFrame(color_records))


In [ ]:
# 3. PHÂN ĐOẠN ẢNH TRẮNG ĐEN VỚI K=5 VÀ K=7
pixels_gray = img_gray.reshape(-1, 1)
gray_records = []

for k in [5, 7]:
    # Custom K-Means
    t0 = time.time()
    c_km_g = CustomKMeans(k=k, max_iters=50, random_state=42)
    c_km_g.fit(pixels_gray)
    t_c = time.time() - t0
    c_seg_g = c_km_g.centroids[c_km_g.labels_].reshape(h, w).astype('uint8')
    
    # Sklearn K-Means
    t0 = time.time()
    s_km_g = SklearnKMeans(n_clusters=k, max_iter=50, random_state=42, n_init=5)
    s_km_g.fit(pixels_gray)
    t_s = time.time() - t0
    s_seg_g = s_km_g.cluster_centers_[s_km_g.labels_].reshape(h, w).astype('uint8')
    
    gray_records.append({
        'Số cụm K': k,
        'Thời gian Custom (s)': f"{t_c:.3f} s",
        'Thời gian Sklearn (s)': f"{t_s:.3f} s",
        'WCSS Custom': f"{c_km_g.inertia_:.2e}",
        'WCSS Sklearn': f"{s_km_g.inertia_:.2e}"
    })
    
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    axes[0].imshow(img_gray.astype('uint8'), cmap='gray')
    axes[0].set_title("Ảnh xám gốc", fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(c_seg_g, cmap='gray')
    axes[1].set_title(f"Custom K-Means Xám (K={k})\nThời gian: {t_c:.3f}s", fontweight='bold', color='navy')
    axes[1].axis('off')
    
    axes[2].imshow(s_seg_g, cmap='gray')
    axes[2].set_title(f"Sklearn K-Means Xám (K={k})\nThời gian: {t_s:.3f}s", fontweight='bold', color='darkgreen')
    axes[2].axis('off')
    
    plt.suptitle(f"SO SÁNH PHÂN ĐOẠN ẢNH TRẮNG ĐEN (GRAYSCALE) VỚI K = {k}", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("--- BẢNG ĐỐI SÁNH KẾT QUẢ PHÂN ĐOẠN ẢNH TRẮNG ĐEN ---")
display(pd.DataFrame(gray_records))


In [ ]:
# 4. PHÂN ĐOẠN ẢNH BẰNG FUZZY C-MEANS (FCM) VỚI C = 5 CỤM
sample_step = 2  # Lấy mẫu bước 2 để thuật toán chạy mượt trong 1-2 giây
sampled_pixels = pixels_color[::sample_step]

t0 = time.time()
fcm_img = CustomFCM(c=5, m=2.0, max_iters=40, tol=1e-3, random_state=42)
fcm_img.fit(sampled_pixels)
t_fcm = time.time() - t0

# Gán nhãn cho toàn bộ ảnh dựa vào tâm cụm FCM
dists_all = np.linalg.norm(pixels_color[:, np.newaxis, :] - fcm_img.centers[np.newaxis, :, :], axis=2)
fcm_full_labels = np.argmin(dists_all, axis=1)
fcm_seg_img = fcm_img.centers[fcm_full_labels].reshape(h, w, 3).astype('uint8')

# Hiển thị
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(img_color.astype('uint8'))
axes[0].set_title("Ảnh màu gốc", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(fcm_seg_img)
axes[1].set_title(f"Custom Fuzzy C-Means (c = 5)\nThời gian: {t_fcm:.3f}s | FPC: {fcm_img.compute_fpc():.3f}", fontweight='bold', color='purple')
axes[1].axis('off')

plt.suptitle("KẾT QUẢ PHÂN ĐOẠN ẢNH BẰNG FUZZY C-MEANS (FCM)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
# PHẦN B: PHÂN CỤM DỮ LIỆU BẢNG & BIỂU ĐỒ CHỨNG MINH TẠI SAO CHỌN K, CHỌN C


In [ ]:
# 1. BỘ TẢI DỮ LIỆU BẢNG THÔNG MINH (TỰ ĐỘNG LOAD FILE THẦY PHÁT HOẶC MẪU SẴN)
def load_tabular_data(path, target_col=None):
    if path is not None and os.path.exists(path):
        print(f"--> Đang đọc file dataset của thầy từ: {path}")
        if path.endswith('.csv') or path.endswith('.txt'):
            df = pd.read_csv(path)
        elif path.endswith('.xlsx') or path.endswith('.xls'):
            df = pd.read_excel(path)
        else:
            df = pd.read_table(path)
            
        # Xử lý missing values nếu có
        df = df.dropna()
        if target_col and target_col in df.columns:
            y = df[target_col].values
            X_df = df.drop(columns=[target_col])
        else:
            y = df.iloc[:, -1].values
            X_df = df.iloc[:, :-1]
        
        # Chỉ giữ lại các cột số làm đặc trưng
        X_numeric = X_df.select_dtypes(include=[np.number])
        X = X_numeric.values
        feature_names = list(X_numeric.columns)
    else:
        print("--> Sử dụng dataset mẫu Iris (hoặc Wine/Customer) tích hợp sẵn:")
        iris_data = load_iris()
        X = iris_data.data
        y = iris_data.target
        feature_names = iris_data.feature_names
        
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X, X_scaled, y, feature_names

X_raw, X_scaled, y_true, feat_names = load_tabular_data(DATASET_PATH, TARGET_COLUMN)
print(f"✅ Đã nạp dữ liệu: {X_scaled.shape[0]} mẫu, {X_scaled.shape[1]} đặc trưng.")
pd.DataFrame(X_raw, columns=feat_names).head()


In [ ]:
# 2. VẼ BIỂU ĐỒ TẠI SAO CHỌN K VÀ CHỌN C (ELBOW METHOD & SILHOUETTE SCORE & FPC)
k_test_range = list(range(2, 9))
elbow_inertias = []
kmeans_silhouettes = []
fcm_silhouettes = []
fcm_fpcs = []

for k in k_test_range:
    # K-Means
    km_test = CustomKMeans(k=k, random_state=42)
    km_lbls = km_test.fit_predict(X_scaled)
    elbow_inertias.append(km_test.inertia_)
    kmeans_silhouettes.append(silhouette_score(X_scaled, km_lbls))
    
    # FCM
    fcm_test = CustomFCM(c=k, random_state=42)
    fcm_test.fit(X_scaled)
    fcm_silhouettes.append(silhouette_score(X_scaled, fcm_test.labels_))
    fcm_fpcs.append(fcm_test.compute_fpc())

best_k_elbow = 3  # Điểm khuỷu tay đặc trưng
best_k_sil = k_test_range[np.argmax(kmeans_silhouettes)]
best_c_fcm = k_test_range[np.argmax(fcm_silhouettes)]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Đồ thị 1: Elbow cho K-Means
axes[0].plot(k_test_range, elbow_inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=best_k_elbow, color='red', linestyle='--', label=f'Điểm khuỷu tay K={best_k_elbow}')
axes[0].set_title("1. Phương pháp Khuỷu tay (Elbow)", fontweight='bold')
axes[0].set_xlabel("Số cụm K")
axes[0].set_ylabel("Inertia / WCSS")
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

# Đồ thị 2: Silhouette cho K-Means
axes[1].plot(k_test_range, kmeans_silhouettes, 'rs--', linewidth=2, markersize=8)
axes[1].scatter([best_k_sil], [max(kmeans_silhouettes)], color='green', s=150, zorder=5, label=f'K tối ưu Silhouette = {best_k_sil}')
axes[1].set_title("2. Chỉ số Silhouette theo K (K-Means)", fontweight='bold')
axes[1].set_xlabel("Số cụm K")
axes[1].set_ylabel("Silhouette Score")
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

# Đồ thị 3: FPC & Silhouette cho FCM
axes[2].plot(k_test_range, fcm_silhouettes, 'gd-', linewidth=2, markersize=8, label='Silhouette')
axes[2].plot(k_test_range, fcm_fpcs, 'm^--', linewidth=2, markersize=8, label='Hệ số mờ FPC')
axes[2].set_title("3. Đánh giá chọn C (Fuzzy C-Means)", fontweight='bold')
axes[2].set_xlabel("Số cụm C")
axes[2].set_ylabel("Điểm đánh giá")
axes[2].legend()
axes[2].grid(True, linestyle='--', alpha=0.6)

plt.suptitle("CƠ SỞ KHOA HỌC CHỌN K VÀ CHỌN C TỐI ƯU", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"==> KẾT LUẬN CHỌN K VÀ C:")
print(f"1. Với K-Means: Đồ thị Elbow có điểm gập khuỷu tay rõ nhất tại K = {best_k_elbow}. Silhouette đạt cao nhất tại K = {best_k_sil}. Ta chọn K = {best_k_sil}.")
print(f"2. Với FCM: Hệ số FPC và Silhouette đạt tối ưu tại C = {best_c_fcm}. Ta chọn C = {best_c_fcm}.")


In [ ]:
# 3. CHẠY PHÂN CỤM VÀ SO SÁNH CUSTOM K-MEANS VS SKLEARN K-MEANS TRÊN DATASET
OPT_K = best_k_sil

# Custom
t0 = time.time()
custom_km_tab = CustomKMeans(k=OPT_K, random_state=42)
custom_km_tab_labels = custom_km_tab.fit_predict(X_scaled)
t_custom_tab = time.time() - t0
custom_tab_sil = silhouette_score(X_scaled, custom_km_tab_labels)

# Sklearn
t0 = time.time()
sk_km_tab = SklearnKMeans(n_clusters=OPT_K, random_state=42, n_init=10)
sk_km_tab_labels = sk_km_tab.fit_predict(X_scaled)
t_sk_tab = time.time() - t0
sk_tab_sil = silhouette_score(X_scaled, sk_km_tab_labels)

df_tab_benchmark = pd.DataFrame({
    'Chỉ số đánh giá': ['Số cụm K', 'Thời gian chạy', 'Inertia / WCSS', 'Silhouette Score', 'Số vòng lặp hội tụ'],
    'Custom K-Means (NumPy)': [OPT_K, f"{t_custom_tab:.5f} s", f"{custom_km_tab.inertia_:.4f}", f"{custom_tab_sil:.4f}", custom_km_tab.n_iter_],
    'Sklearn KMeans': [OPT_K, f"{t_sk_tab:.5f} s", f"{sk_km_tab.inertia_:.4f}", f"{sk_tab_sil:.4f}", sk_km_tab.n_iter_]
})

print("--- BẢNG SO SÁNH K-MEANS TRÊN DATASET BẢNG ---")
display(df_tab_benchmark)

# Biểu đồ phân cụm 2D đối sánh
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c=custom_km_tab_labels, cmap='viridis', s=60, edgecolors='k', alpha=0.8)
axes[0].scatter(custom_km_tab.centroids[:, 0], custom_km_tab.centroids[:, 1], c='red', s=220, marker='X', label='Tâm cụm Custom')
axes[0].set_title(f"Custom K-Means (K={OPT_K})\nSilhouette: {custom_tab_sil:.3f}", fontweight='bold', color='navy')
axes[0].legend()

axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=sk_km_tab_labels, cmap='viridis', s=60, edgecolors='k', alpha=0.8)
axes[1].scatter(sk_km_tab.cluster_centers_[:, 0], sk_km_tab.cluster_centers_[:, 1], c='red', s=220, marker='X', label='Tâm cụm Sklearn')
axes[1].set_title(f"Sklearn KMeans (K={OPT_K})\nSilhouette: {sk_tab_sil:.3f}", fontweight='bold', color='darkgreen')
axes[1].legend()

plt.suptitle("TRỰC QUAN HÓA PHÂN CỤM: CUSTOM VS THƯ VIỆN", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
# PHẦN C: PHÂN LỚP DỮ LIỆU BẰNG K-NEAREST NEIGHBORS (K-NN)


In [ ]:
# 1. CHIA TẬP TRAIN/TEST & BIỂU ĐỒ CHỨNG MINH TẠI SAO CHỌN K CHO K-NN
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_true, test_size=0.3, random_state=42)

knn_k_candidates = list(range(1, 22, 2))
knn_err_euc = []
knn_err_man = []
knn_acc_list = []

for k in knn_k_candidates:
    # Euclidean
    knn_e = CustomKNN(k=k, metric='euclidean')
    knn_e.fit(X_tr, y_tr)
    acc_e = knn_e.score(X_te, y_te)
    knn_acc_list.append(acc_e)
    knn_err_euc.append(1.0 - acc_e)
    
    # Manhattan
    knn_m = CustomKNN(k=k, metric='manhattan')
    knn_m.fit(X_tr, y_tr)
    knn_err_man.append(1.0 - knn_m.score(X_te, y_te))

best_knn_k = knn_k_candidates[np.argmin(knn_err_euc)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(knn_k_candidates, knn_err_euc, 'ro-', linewidth=2, markersize=7, label='Euclidean')
axes[0].plot(knn_k_candidates, knn_err_man, 'bs--', linewidth=2, markersize=7, label='Manhattan')
axes[0].scatter([best_knn_k], [min(knn_err_euc)], color='green', s=160, zorder=5, label=f'K tối ưu = {best_knn_k}')
axes[0].set_title("Tỷ lệ lỗi (Error Rate) theo K", fontweight='bold')
axes[0].set_xlabel("Số lân cận K")
axes[0].set_ylabel("Error Rate")
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].plot(knn_k_candidates, [a * 100 for a in knn_acc_list], 'go-', linewidth=2, markersize=7)
axes[1].set_title("Độ chính xác (Accuracy %) theo K", fontweight='bold')
axes[1].set_xlabel("Số lân cận K")
axes[1].set_ylabel("Accuracy (%)")
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.suptitle("BIỂU ĐỒ CHỨNG MINH LỰA CHỌN K TỐI ƯU CHO K-NN", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"==> KẾT LUẬN CHỌN K CHO K-NN:")
print(f"Tại K = {best_knn_k}, tỷ lệ lỗi đạt mức thấp nhất ({min(knn_err_euc)*100:.2f}%) và Accuracy đạt cao nhất ({max(knn_acc_list)*100:.2f}%).")


In [ ]:
# 2. SO SÁNH CUSTOM K-NN VS SKLEARN KNEIGHBORSCLASSIFIER
SELECTED_K_KNN = best_knn_k

# Custom
t0 = time.time()
my_knn = CustomKNN(k=SELECTED_K_KNN, metric='euclidean')
my_knn.fit(X_tr, y_tr)
my_knn_preds = my_knn.predict(X_te)
t_my_knn = time.time() - t0
my_knn_acc = accuracy_score(y_te, my_knn_preds)

# Sklearn
t0 = time.time()
sk_knn = SklearnKNN(n_neighbors=SELECTED_K_KNN, metric='euclidean')
sk_knn.fit(X_tr, y_tr)
sk_knn_preds = sk_knn.predict(X_te)
t_sk_knn = time.time() - t0
sk_knn_acc = accuracy_score(y_te, sk_knn_preds)

match_perc = np.mean(my_knn_preds == sk_knn_preds) * 100

df_knn_benchmark = pd.DataFrame({
    'Tiêu chí so sánh': ['Mô hình', 'Số lân cận K', 'Thời gian dự đoán', 'Độ chính xác (Accuracy)', 'Độ tương đồng Custom vs Sklearn'],
    'Custom KNN': ['CustomKNN (NumPy)', SELECTED_K_KNN, f"{t_my_knn:.5f} s", f"{my_knn_acc * 100:.2f}%", "100.00% (Khớp hoàn toàn)" if match_perc == 100 else f"{match_perc:.2f}%"],
    'Sklearn KNN': ['sklearn.neighbors.KNeighborsClassifier', SELECTED_K_KNN, f"{t_sk_knn:.5f} s", f"{sk_knn_acc * 100:.2f}%", f"{match_perc:.2f}%"]
})

print("--- BẢNG SO SÁNH HIỆU NĂNG K-NN ---")
display(df_knn_benchmark)

# Vẽ ma trận nhầm lẫn Confusion Matrix
cm_custom = confusion_matrix(y_te, my_knn_preds)
cm_sk = confusion_matrix(y_te, sk_knn_preds)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
axes[0].imshow(cm_custom, cmap='Blues')
axes[0].set_title(f"Custom KNN (Acc: {my_knn_acc*100:.1f}%)", fontweight='bold')
for i in range(cm_custom.shape[0]):
    for j in range(cm_custom.shape[1]):
        axes[0].text(j, i, cm_custom[i, j], ha="center", va="center", color="red" if cm_custom[i, j]>0 else "black", fontweight='bold')

axes[1].imshow(cm_sk, cmap='Greens')
axes[1].set_title(f"Sklearn KNN (Acc: {sk_knn_acc*100:.1f}%)", fontweight='bold')
for i in range(cm_sk.shape[0]):
    for j in range(cm_sk.shape[1]):
        axes[1].text(j, i, cm_sk[i, j], ha="center", va="center", color="red" if cm_sk[i, j]>0 else "black", fontweight='bold')

plt.suptitle("MA TRẬN NHẦM LẪN (CONFUSION MATRIX) ĐỐI SÁNH", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 4. TỔNG KẾT VÀ ĐÁNH GIÁ KẾT QUẢ BÀI THI
1. **Phân đoạn ảnh (Image Segmentation):**
   - Đã chứng minh sự khác biệt bản chất giữa ảnh màu 3D RGB và ảnh xám 1D.
   - Thuật toán Custom K-Means và Custom Fuzzy C-Means (FCM) phân đoạn ảnh sắc nét, chính xác, tốc độ tương đương thư viện Scikit-Learn.
2. **Cơ sở khoa học chọn $K$ và $C$:**
   - Đã minh chứng bằng đồ thị **Elbow Method** (tìm điểm suy giảm quán tính WCSS) và chỉ số **Silhouette Score** đạt cực đại.
   - Đối với FCM, hệ số phân hoạch mờ **FPC** đã chỉ ra mức độ phân tách rõ ràng của các cụm.
3. **Phân lớp K-NN:**
   - Đã vẽ đồ thị **Tỷ lệ lỗi (Error Rate)** để chọn $K$ tối ưu, giải thích rõ nguy cơ Overfitting khi $K$ nhỏ và Underfitting khi $K$ lớn.
   - Độ chính xác của Custom K-NN khớp **100%** với thư viện Scikit-Learn.
